#  TESTING  LDM COMPONENTS

In [13]:
from transformers import CLIPTokenizer,CLIPTextModel
from diffusers import AutoencoderKL
import torch
import torch.nn  as nn
  
class Clip_VAE(nn.Module):
    def __init__(self,
                model_name:  str,
                device,
                tokenizer = None,
                text_encoder = None,
                Vae = None 
                 ):
        """Module for clip tokenizer,text_encoder and vae decoder/encoder

        Attributes:
            text (str): specify component type(CLIP / VAE)
            device (cuda/cpu): device
            tokenizer : CLIPTokenizer, for input_ids or tokens
            text_encoder : CLIPTextModel, to  encode ids or  tokens to embeddings
            Vae : AutoencoderKL
        """
        super().__init__()
        self.model_name =  model_name
        self.device = device
        self.tokenizer  = tokenizer
        self.text_encoder = text_encoder
        self.Vae = Vae
        
        if self.model_name =='clip':
            self.get_tokens = self.tokenizer.from_pretrained('CompVis/stable-diffusion-v1-4',subfolder='tokenizer')
            self.get_embeddings = self.text_encoder.from_pretrained('CompVis/stable-diffusion-v1-4',subfolder='text_encoder').to(self.device)
            
        if  self.model_name == 'Vae_encode' or self.model_name == 'Vae_decode':
            self.latent_scaling_factor =  0.18215
            self.autoencoder = self.Vae.from_pretrained('CompVis/stable-diffusion-v1-4',subfolder='vae').to(device)
    @torch.inference_mode()        
    def forward(self,input : str | torch.Tensor)-> torch.Tensor:
        if self.model_name=='clip':
            input_ids = self.get_tokens([input],
                                       padding='max_length',
                                       max_length = 77,
                                       truncation=True, 
                                       return_tensors="pt",
                                       )['input_ids'] 
            input_ids_tensor =  input_ids.to(self.device)  
            text_embeddings =   self.get_embeddings(input_ids_tensor)['last_hidden_state']
            return  text_embeddings
        input = input.to(self.device)
        if  self.model_name == 'Vae_encode':
            encoded = self.latent_scaling_factor * self.autoencoder.encode(input).sample
            return encoded
        if  self.model_name == 'Vae_decode':
            decoded  = self.autoencoder.decode(input/self.latent_scaling_factor).sample
            return (decoded/2 + 0.5).clamp(0,1) # [-1,1] to [0,1]

In [14]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [15]:
text = 'i am a boy'
encode_text = Clip_VAE('clip',device,CLIPTokenizer,CLIPTextModel)
encode_text(text).shape

torch.Size([1, 77, 768])

In [16]:
l = torch.randn(1,4,64,64)
vae =  Clip_VAE('Vae_decode',device,Vae=AutoencoderKL)
vae(l).shape

torch.Size([1, 3, 512, 512])

In [18]:
## UNEt
import torch
import torch.nn  as nn
class  TimestepEmbedding(nn.Module):
    """Encodes scalar diffusion steps into continuous vectors.
    """
    def  __init__(self,base_dim,f_dim):
        super().__init__()
        self.base_dim = base_dim # or d_model
        self.f_dim=  f_dim
        
        self.mlp = nn.Sequential(
            nn.Linear(self.base_dim,self.f_dim),
            nn.SiLU(),
            nn.Linear(self.f_dim,self.f_dim)
        )
    def forward(self,timesteps: torch.Tensor)-> torch.Tensor:
        time_vector = timesteps.flatten().float()#(len(steps),)
        b =  time_vector.shape[0]
        dim_vector = torch.arange(0,self.base_dim,2,device=timesteps.device).float()#(base_dim/2,)
        frequency_scales = 10000 ** (dim_vector/self.base_dim)
        f_s =  1/frequency_scales
        #final = time_vector[:,None] @ f_s[None,:]  # (steps,1) x (1,base_dim/2)
        final = torch.outer(time_vector,f_s)#(steps,base_dim/2)
        embedding = torch.zeros(b,self.base_dim,device=timesteps.device) #(steps,base_dim)
        embedding[:,0::2] = torch.sin(final)
        embedding[:,1::2]  = torch.cos(final)
        context =  self.mlp(embedding)  #(steps,final dim)
        return context

In [19]:
t  = torch.arange(0,1000,1)
embed =  TimestepEmbedding(320,1280)
embed(t).shape

torch.Size([1000, 1280])

In [20]:
def Normalize(in_channels, num_groups=32):
    return torch.nn.GroupNorm(num_groups=num_groups, num_channels=in_channels, eps=1e-6, affine=True)

class ResBlock(nn.Module):
    """Processes spatial image features and injects time context.

    Attributes:
        in_ch : input channel dim
        out_ch : output channel dim
    """
    def __init__(self,in_ch,d_t_embed,dropout=0.,out_ch=None):
        super().__init__()
        self.embed_dim = d_t_embed
        self.in_channels = in_ch
        self.out_channels = out_ch  if out_ch is not None  else  in_ch
        self.in_layers = nn.Sequential(
                Normalize(in_channels=self.in_channels),
                nn.SiLU(),
                nn.Conv2d(
                        self.in_channels,
                        self.out_channels,
                        kernel_size=3,
                        padding=1,
                    )
        )
        
        # Project time embeddings
        self.emb_layers = nn.Sequential(
            nn.SiLU(),
            nn.Linear(self.embed_dim,self.out_channels)
            
            )
        self.out_layers = nn.Sequential(
                Normalize(in_channels=self.out_channels),
                nn.SiLU(),
                nn.Dropout(dropout),
                nn.Conv2d(
                        self.out_channels,
                        self.out_channels,
                        kernel_size=3,
                        padding=1,
                    )
        )
        
        if self.in_channels == self.out_channels:
            self.skip_connection = nn.Identity()
        else:
            self.skip_connection = nn.Conv2d(
                                    self.in_channels,
                                    self.out_channels, 
                                    1
                                    )    
    def forward(self,x,time_context):
        """
            (x)Image feats: (Batch, In_Channels, H, W)
            Time context: (Batch, 1280)
        """
        h = x
        h =  self.in_layers(x)
        # project time context
        proj  = self.emb_layers(time_context)[:,:,None,None] #(b,feats,1,1)
        h =  h + proj
        h =  self.out_layers(h)
        return self.skip_connection(x) + h
        

In [21]:
block = ResBlock(in_ch=320,d_t_embed=1280, out_ch=None,dropout=0.1)
mock_latents = torch.randn(2, 320, 64, 64)       # Shape: [Batch, Channels, H, W]
mock_time_emb = torch.randn(2, 1280)             # Shape: [Batch, Time_Dim]
output = block(mock_latents, mock_time_emb)
print("Output Shape:", output.shape)


Output Shape: torch.Size([2, 320, 64, 64])


In [22]:
import torch.nn.functional as F

class CrossAttn(nn.Module):
    def __init__(self, query_dim, context_dim=None, heads=8, dim_head=64, dropout=0.):
        super().__init__()
        context_dim = context_dim  if context_dim is not None else query_dim
        # This tells the linear layers how big the shared attention space needs to be.
        inner_dim = dim_head * heads
        self.heads = heads
        self.scale = dim_head ** -0.5
        
        self.to_q = nn.Linear(query_dim, inner_dim, bias=False)
        self.to_k = nn.Linear(context_dim, inner_dim, bias=False)
        self.to_v = nn.Linear(context_dim, inner_dim, bias=False)
        self.to_out = nn.Sequential(
            nn.Linear(inner_dim, query_dim),
            nn.Dropout(dropout)
        )
    def forward(self,x,context_matrix=None):
        '''
        Image sequence(x) : [b,pixels,c]
         context  matrix : [b,seq,context_dim]
        '''
        if context_matrix is None:
            context_matrix = x
        b,pixels,_ = x.size()
        _,context,_ = context_matrix.size()
        q  = self.to_q(x) #(b,pixels,inner_dim)
        k = self.to_k(context_matrix)#(b,seq_k,inner_dim) 
        v = self.to_v(context_matrix)#(b,seq_v,inner_dim)
        
        #Merge heads into the batch dimension # (b*h,pixels/context,dim_head)
        dim_head = q.shape[-1]//self.heads
        q  = q.view(b,pixels,self.heads,dim_head).permute(0,2,1,3).reshape(b*self.heads,pixels,dim_head)
        k  = k.view(b,context,self.heads,dim_head).permute(0,2,1,3).reshape(b*self.heads,context,dim_head)
        v  = v.view(b,context,self.heads,dim_head).permute(0,2,1,3).reshape(b*self.heads,context,dim_head)
        scores = torch.bmm(q, k.transpose(-2, -1)) * self.scale #(b*h,pixels,context)
        attn = F.softmax(scores, dim=-1) # (b*h,pixels,context)
        
        res = attn @ v #(b*h,pixels,dim_head)
        res = res.view(b,self.heads,pixels,dim_head).permute(0,2,1,3).reshape(b,pixels,-1) #(b,pixels,inner_dim)
        return self.to_out(res) #(b,pixels,query_dim)
        

In [23]:
batch_size = 2
pixels = 64 * 64     # 4096 flattened pixel tokens (from a 64x64 latent)
text_len = 77        # Fixed token sequence window from tokenizer

mock_image_sequence = torch.randn(batch_size, pixels, 640) # Shape: [2, 4096, 640]
mock_text_sequence = torch.randn(batch_size, text_len, 768) # Shape: [2, 77, 768]


# Cross-Attention  (Text Fusion)
cross_attn = CrossAttn(query_dim=640, context_dim=768)
output_cross = cross_attn(mock_image_sequence, context_matrix=mock_text_sequence)
print("Cross-Attention Output Shape:", output_cross.shape) #[2, 4096, 640]

# Self-Attention  (Image Context)
self_attn = CrossAttn(query_dim=640)
output_self = self_attn(mock_image_sequence, context_matrix=None)
print("Self-Attention Output Shape: ", output_self.shape) #[2, 4096, 640]


Cross-Attention Output Shape: torch.Size([2, 4096, 640])
Self-Attention Output Shape:  torch.Size([2, 4096, 640])


In [24]:
class Upsample(nn.Module):
    def __init__(self,channels,with_conv : bool = True):
        super().__init__()
        self.with_conv  = with_conv
        if self.with_conv:
            self.conv = nn.Conv2d(
                                channels,
                                channels,
                                kernel_size=3,
                                stride = 1,
                                padding = 1
                                ) 
    def forward(self,x):
        x  = F.interpolate(x,scale_factor=2,mode='nearest')
        return self.conv(x) if self.with_conv else x   
   
class Downsample(nn.Module):
    def __init__(self,channels,with_conv :  bool = True):
        super().__init__()
        self.with_conv = with_conv
        if self.with_conv:
            self.conv = nn.Conv2d(
                                channels,
                                channels,
                                kernel_size=3,
                                stride=2,
                                padding=1)
    def forward(self,x):  
        return  self.conv(x) 
     
    

In [25]:
image =  torch.randn(1,3,224,224)
up = Upsample(3)(image)
up.shape 


torch.Size([1, 3, 448, 448])

In [26]:
d = Downsample(3)(image)
d.shape 


torch.Size([1, 3, 112, 112])

In [27]:
from einops import rearrange
class TransformerBlock(nn.Module):
    def __init__(self,channels,n_heads,head_dim,dropout,context_dim=None):
        super().__init__()
        self.in_channel = channels
        inner_dim = int(channels *  4)
        self.norm1 = nn.LayerNorm(channels)
        self.norm2 = nn.LayerNorm(channels)
        self.norm3 = nn.LayerNorm(channels)
        
        # 2 attention  layers and  a feed forward,
        self.attn1 = CrossAttn(query_dim=channels,heads=n_heads,dim_head=head_dim,dropout=dropout) #self attn (context=None)
        self.attn2 = CrossAttn(query_dim=channels,context_dim=context_dim,heads=n_heads,
                               dim_head=head_dim,dropout=dropout) # crossatn with  context
        
        self.ff  = nn.Sequential(
            nn.Linear(channels,inner_dim),
            nn.GELU(),
            nn.Linear(inner_dim,channels)
        )
    def forward(self,x,context=None):
        x = self.attn1(self.norm1(x)) + x
        x = self.attn2(self.norm2(x),context)  + x  #passing context into forward method 
        x = self.ff(self.norm3(x)) + x
        return x
    
class SpatialTransformer(nn.Module):
    def __init__(self,
                channels,
                n_heads,
                head_dim,
                context_dim=None,
                depth = 1,
                dropout=0.1,               
                ):
        super().__init__()
        self.in_channel = channels
        
        inner_dim = int(n_heads * head_dim)
        self.norm = Normalize(channels)
        # 1x 1conv
        self.proj_in =  nn.Conv2d(
                                  channels,
                                  inner_dim,
                                  kernel_size=1 ,
                                  stride=1,
                                  padding=0 
        )
        self.transformer_blocks = nn.ModuleList(
            [TransformerBlock(inner_dim,n_heads,head_dim,dropout,context_dim) for _ in range(depth)]
            )
        self.proj_out =  nn.Conv2d(
                                  inner_dim,
                                  channels,
                                  kernel_size=1  
        )
    
        
    def forward(self,x,context_matrix=None):
        # x :(b,c,h,w)
        # context_matrix : (b,seq,dim)
        b,c,h,w = x.shape
        
        x_in = x
        res = self.norm(x_in)
        res = self.proj_in(res)#(b,c,h,w)
        #1D Sequence Format (Flattening)
        #res =  res.permute(0,2,3,1).reshape(b,h*w,c)
        res = rearrange(res,'b c h w -> b (h w) c')#(Batch, Pixels, Channels)
        for block in self.transformer_blocks:
            res = block(res,context_matrix) # (batch,pixels,channels)
        res = rearrange(res, 'b (h w) c -> b c h w', h=h, w=w)
        res = self.proj_out(res) + x # (b,c,h,w)
        return res         

In [28]:
transformer_block = SpatialTransformer(
    channels=640,
    n_heads=8, # 8
    head_dim=640//8,# dim or channels/num_heads
    context_dim=768    
)
mock_image_features = torch.randn(2, 640, 32, 32) 
mock_text_features = torch.randn(2, 77, 768) 
output_features = transformer_block(mock_image_features, context_matrix=mock_text_features)
print("--- SHAPE RESULTS ---")
print(f"Output Image Tensor Shape: {output_features.shape}")

--- SHAPE RESULTS ---
Output Image Tensor Shape: torch.Size([2, 640, 32, 32])


In [29]:
class TimestepEmbedSequential(nn.Sequential):
    def  forward(self,x,embed,cond=None):
        for layer in self:
            if isinstance(layer,ResBlock):
                x = layer(x,embed)
            elif isinstance(layer,SpatialTransformer):
                x = layer(x,cond)
            else:
                x  = layer(x)
        return x        

In [30]:
from typing import List


class UNetConditional2D(nn.Module):
    """Encoder-decoder network  to  predict the noise  from  latents
    """
    def __init__(self,
                channels : int, # same as first val in block_out_channels
                in_channels : int,
                out_channels : int,
                block_out_channels : List[int], #Channel depth for each of the four resolution stages.
                layers_per_block : int, # number of resnetblocks in each up or down/up block
                levels : int ,# Number of  levels 
                attn_levels : List[int],
                cross_attention_dim : int = 768,
                n_heads  : int = 8
                 ):
        super().__init__()
        
        
        # Time embedding
        self.time_embedding = TimestepEmbedding(channels,channels*4)
        # Encoder
        self.input_blocks = nn.ModuleList()
        self.input_blocks.append(TimestepEmbedSequential(
            nn.Conv2d(in_channels,channels,3,padding=1) # Projecting input tensor
        ))
        input_block_channels = [channels]
        for i in range(levels):
            for  _ in  range(layers_per_block):
                # for each level(down block types ) add resnetblocks (2) ,some attention blocks and downsample at  the end
                #  taking last  element block_out_channels as d_t_embed
                layers = [ResBlock(in_ch=channels,d_t_embed=block_out_channels[-1],out_ch=block_out_channels[i])]
                channels = block_out_channels[i]
                
                if i in attn_levels:
                    layers.append(SpatialTransformer(channels, n_heads, head_dim=80, context_dim=cross_attention_dim)) 
                    
                self.input_blocks.append(TimestepEmbedSequential(*layers))
                input_block_channels.append(channels) # all input block channels  , use later for decoder(skip connections)
                
            if i != levels-1:
                self.input_blocks.append(TimestepEmbedSequential(Downsample(channels)))
                input_block_channels.append(channels)
         
        self.middle_block = TimestepEmbedSequential(
            ResBlock(channels,block_out_channels[-1]),
            SpatialTransformer(channels,n_heads=n_heads,head_dim=80,context_dim=cross_attention_dim),
            ResBlock(channels,block_out_channels[-1]),
        )   
        # (decoder) 
        self.output_blocks = nn.ModuleList()
        for i in reversed(range(levels)):
            for j in range(layers_per_block+1):
                #skip connection at the resnet block
                layers = [ResBlock(in_ch=channels + input_block_channels.pop(),d_t_embed=block_out_channels[-1],out_ch=block_out_channels[i])]
                channels = block_out_channels[i]
                if i in attn_levels:
                    layers.append(SpatialTransformer(channels,n_heads,head_dim=80,context_dim=cross_attention_dim))
                if i != 0 and j == layers_per_block:
                    layers.append(Upsample(channels))    
                self.output_blocks.append(TimestepEmbedSequential(*layers))   
                
        self.out = nn.Sequential(
            nn.GroupNorm(32,channels),
            nn.SiLU(),
            nn.Conv2d(channels, out_channels, 3, padding=1),
        )    
        
        
    def forward(self, x : torch.Tensor, t_steps : torch.Tensor, cond : torch.Tensor):
        input_block = []
        t_embedding  = self.time_embedding(t_steps) # (b,embed_dim)
        for m in  self.input_blocks:
            x =  m(x, t_embedding,cond) # input  modules and  what  each takes
            input_block.append(x)
        x = self.middle_block(x, t_embedding, cond)   
        for m in self.output_blocks:
            x = torch.cat([x,input_block.pop()],dim=1) # skip connections in  the  unet for decoder side(u)
            x  = m(x,t_embedding,cond)
        return  self.out(x)    
            
        


In [31]:
unet = UNetConditional2D(
    channels = 320,
    in_channels = 4,
    out_channels = 4,
    block_out_channels = [320, 640, 1280, 1280],
    layers_per_block = 2,
    levels =  4,
    attn_levels = [0, 1, 2, 3],    
    ) 


In [ ]:
x = torch.randn(1, 4, 64, 64)  # input latent
t = torch.randint(0, 1000, (5,)) # timestep
ctx = torch.randn(1, 77, 768)   # text embeddings

output = unet(x, t, ctx).to(device)



: 

In [ ]:
output.shape

torch.Size([1, 4, 64, 64])

In [ ]:
unet

NameError: name 'unet' is not defined